In [1]:
import lancedb
from lancedb.pydantic import LanceModel, Vector



# get test 10-K data
from equibot.config import settings
import edgar
edgar.set_identity(settings.sec_api_username)
tndm = edgar.Company("TNDM")
tenk = tndm.get_filings(form="10-K")[0].obj()

c:\Users\benak\repos\EquiBot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# initialize db
uri = settings.data_dir + "/equity_research_db"
db = lancedb.connect(uri)  # should create if none exists

In [4]:
tndm._cik

1438133

In [72]:
import json
from typing import Optional, Any
from pydantic import field_validator
from lancedb.pydantic import LanceModel, Vector

class EarningsArtifact(LanceModel):
    id: str                # Unique ID (LlamaIndex node_id)
    vector: Vector(1536)   # Matches text-embedding-3-small
    text: str
    
    cik: str
    ticker: str
    company_name: str
    source_date: str
    form_type: str
    item_name: str
    item_number: Optional[str] = None 
    fiscal_year: int
    fiscal_quarter: int

    # Define as strings so Apache Arrow (LanceDB) accepts them
    metadata: str 
    _node_content: str

    @classmethod
    @field_validator("metadata", "_node_content", mode="before")
    def serialize_dict(cls, v: Any) -> str:
        """Automatically converts dict to JSON string if LlamaIndex sends a dict."""
        if isinstance(v, dict):
            return json.dumps(v)
        return str(v)
    

# create table
table = db.create_table("earnings_artifacts", schema=EarningsArtifact, mode="overwrite")

In [74]:
from equibot.parsers.transcript import TranscriptScraper
from dataclasses import dataclass

@dataclass
class Artifact(edgar.Company):
    """General artifact class for managing metadata"""
    ticker: str
    fiscal_year: int
    fiscal_quarter: int
    
    def __post_init__(self):
        super().__init__(self.ticker.upper())
        
        self.file_obj = self.get_filings(
            form="10-K" if self.fiscal_quarter == 4 else "10-Q",
            year=self.fiscal_year
        )[0].obj()

        self.metadata = {
            "cik": self._cik,
            "ticker": self.ticker,
            "company_name": self.file_obj.company,
            "source_date": self.file_obj.filing_date.strftime("%Y-%m-%d"),
            "form_type": self.file_obj.form,
            "fiscal_year": self.fiscal_year,
            "fiscal_quarter": self.fiscal_quarter
        }

        # only appropriate for 10-Ks atm!!!!!
        tenk_docs = [
            {"name": "business", "number": "Item 1"},
            {"name": "risk_factors", "number": "Item 1A"},
            {"name": "management_discussion", "number": "Item 7"},
        ]
        self.documents = {
            d['name']: Document(
                text=getattr(self.file_obj, d['name'], None),
                metadata=self.metadata | {"item_name": d['name'], "item_number": d['number']}
            )
            for d in tenk_docs
        }

        self.documents["earnings_call_transcript"] = Document(
            text=self.get_transcript(),
            metadata=self.metadata | {"item_name": "earnings_call_transcript", "item_number": None}
        )


    def get_transcript(self):
        """Helper method to scrape earnings call transcript from YFinance"""
        try:
            ec_scraper = TranscriptScraper(ticker=self.ticker)
            transcript_data = ec_scraper.get_transcript(quarter=self.fiscal_quarter, year=self.fiscal_year)
            transcript = "\n".join([
                f"{chunk['speaker']}, {chunk['title']} [{chunk['time']}]\n" + chunk['text'] + "\n"
                for chunk in transcript_data
            ])

        except Exception as e:
            print(f"No transcript retrieved. Error raised: {e}")
        
        return transcript

In [73]:
from llama_index.core import Document, StorageContext, VectorStoreIndex
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.vector_stores.lancedb import LanceDBVectorStore


# initialize LanceDB vector store
vector_store = LanceDBVectorStore(
    uri=uri,
    table_name="earnings_artifacts",
    flat_metadata=True,  # ensures LlamaIndex treats metadata as columns in the LanceDB table
    doc_id_key="id"
)

# set up semantic splitter and embedder
embedder = OpenAIEmbedding(
    model="text-embedding-3-small",
    api_key=settings.openrouter_api_key,
    api_base="https://openrouter.ai/api/v1",
)

splitter = SemanticSplitterNodeParser(
    buffer_size=1, 
    breakpoint_percentile_threshold=95, 
    embed_model=embedder
)

In [76]:
hmm = Artifact("TNDM", 2025, 4)

In [ ]:
nodes = splitter.get_nodes_from_documents(list(hmm.documents.values()))

In [79]:
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(
    list(hmm.documents.values()),
    storage_context=storage_context,
    transformations=[splitter],
    embed_model=embedder
)

RuntimeError: lance error: LanceError(Arrow): C Data interface error: NotImplemented: Unsupported cast from struct<_node_content: string, _node_type: string, cik: int64, company_name: string, doc_id: string, document_id: string, fiscal_quarter: int64, fiscal_year: int64, form_type: string, item_name: string, item_number: string, ref_doc_id: string, source_date: string, ticker: string> to utf8 using function cast_string. Detail: Python exception: Traceback (most recent call last):
  File "c:\Users\benak\repos\EquiBot\.venv\Lib\site-packages\lancedb\table.py", line 315, in gen
    yield pa.Table.from_batches([batch]).cast(reordered_schema).to_batches()[0]
          ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^
  File "pyarrow/table.pxi", line 4728, in pyarrow.lib.Table.cast
    casted = column.cast(field.type, safe=safe, options=options)
  File "pyarrow/table.pxi", line 597, in pyarrow.lib.ChunkedArray.cast
    return _pc().cast(self, target_type, safe=safe, options=options)
  Fi

In [10]:
import tiktoken
tokenizer = tiktoken.get_encoding("cl100k_base")  # tokenizer for embed-3-small
print(len(tenk.management_discussion))
print(len(tokenizer.encode(tenk.management_discussion)))

40343
8068


In [82]:
help(tenk)

Help on TenK in module edgar.company_reports.ten_k object:

class TenK(edgar.company_reports._base.CompanyReport)
 |  TenK(filing)
 |
 |  Method resolution order:
 |      TenK
 |      edgar.company_reports._base.CompanyReport
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __getitem__(self, item_or_part: str)
 |      Get section/item text by name or number.
 |
 |      Supports multiple lookup formats:
 |      - Standard format: 'Item 1', 'Item 1A', 'Item 7'
 |      - Short format: '1', '1A', '7', '7A'
 |      - Friendly names: 'business', 'risk_factors', 'mda'
 |
 |      Falls back to old chunked_document and Cross Reference Index for backward compatibility.
 |
 |      Args:
 |          item_or_part: Section identifier in various formats
 |
 |      Returns:
 |          Section text content as string, or None if not found
 |
 |  __init__(self, filing)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |
 |  __rich__(self)
 |
 |  __str__(self)
 |      Ret

In [84]:
doc = tenk.document

In [86]:
help(doc)

Help on Document in module edgar.documents.document object:

class Document(builtins.object)
 |  Document(root: Node, metadata: DocumentMetadata = <factory>) -> None
 |
 |  Main document class.
 |
 |  Represents a parsed HTML document with methods for content extraction,
 |  search, and transformation.
 |
 |  Methods defined here:
 |
 |  __eq__(self, other)
 |      Return self==value.
 |
 |  __init__(self, root: Node, metadata: DocumentMetadata = <factory>) -> None
 |      Initialize self.  See help(type(self)) for accurate signature.
 |
 |  __iter__(self) -> Iterator[Node]
 |      Iterate over top-level nodes.
 |
 |  __len__(self) -> int
 |      Get number of top-level nodes.
 |
 |  __replace__ = _replace(self, /, **changes) from dataclasses
 |
 |  __repr__(self) -> str
 |      Return repr(self).
 |
 |  chunks(self, chunk_size: int = 512, overlap: int = 128) -> Iterator['DocumentChunk']
 |      Generate document chunks for processing.
 |
 |      Args:
 |          chunk_size: Target ch

In [93]:
doc_txt = doc.text(include_tables=True, include_metadata=False)
print(doc_txt)

UNITED STATES

SECURITIES AND EXCHANGE COMMISSION

WASHINGTON, D. C. 20549

_________________________________________________________________

FORM 10-K

_________________________________________________________________

  ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934  
 ────────────────────────────────────────────────────────────────────────────────────────

For the fiscal year ended December 31, 2025

or

  TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934  
 ────────────────────────────────────────────────────────────────────────────────────────────

For the transition period from to

Commission File Number 001-36189

_________________________________________________________________

Tandem Diabetes Care, Inc.

(Exact name of registrant as specified in its charter)

_________________________________________________________________

  Delaware                                                      20-4327508 

In [102]:
w = doc.walk()

In [119]:
print(next(w))

HeadingNode(id='b5f1e638-2e45-4546-99f0-4393e91f29cc', type=<NodeType.HEADING: 3>, content='FORM 10-K', metadata={'detection_method': 'contextual', 'confidence': 0.6000000000000001}, style=Style(font_size=10.664, font_weight='700', font_style=None, text_align=None, text_decoration=None, color='#000000', background_color=None, margin_top=None, margin_bottom=None, margin_left=None, margin_right=None, padding_top=None, padding_bottom=None, padding_left=None, padding_right=None, display=None, width=None, height=None, line_height=None), semantic_type=None, semantic_role=None, level=3)
